# Part 2 – Air Quality: 01 Ingest

Retrieve PM2.5 measurements (OpenAQ v3) and hourly weather data (Open-Meteo)  
for **New York City**, **Los Angeles**, and **Chicago** over **2025-01-01 → 2025-03-31**.

Raw extracts are saved to `data/raw/openaq/` and `data/raw/openmeteo/`.

In [1]:
import json
import os
import time
import requests
from pathlib import Path

In [2]:
# ── Paths ────────────────────────────────────────────────────────────────────
ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'part2_airquality' else Path(os.getcwd())
RAW_AQ   = ROOT / 'data' / 'raw' / 'openaq'
RAW_WX   = ROOT / 'data' / 'raw' / 'openmeteo'
RAW_AQ.mkdir(parents=True, exist_ok=True)
RAW_WX.mkdir(parents=True, exist_ok=True)

DATE_FROM = '2025-01-01'
DATE_TO   = '2025-03-31'

## 1. OpenAQ v3 – Daily PM2.5 Measurements

**API base:** `https://api.openaq.org/v3`

We query the `/v3/sensors/{sensor_id}/measurements` endpoint with  
`period_name=day` to retrieve pre-aggregated daily summaries.  
The sensor IDs below were looked up via `/v3/locations?city=...&parameter=pm25`.

In [3]:
# OpenAQ v3 – city → sensor_id mapping for PM2.5
# Sensor IDs obtained by querying /v3/locations?city=<city>&country_id=US&parameter=pm25
OPENAQ_BASE = 'https://api.openaq.org/v3'

CITIES_AQ = {
    'nyc':     {'sensor_id': 3140,  'label': 'New York City'},
    'la':      {'sensor_id': 4913,  'label': 'Los Angeles'},
    'chicago': {'sensor_id': 12863, 'label': 'Chicago'},
}

HEADERS = {'accept': 'application/json'}

In [4]:
def fetch_openaq_daily(sensor_id: int, date_from: str, date_to: str) -> dict:
    """Fetch daily PM2.5 measurements from OpenAQ v3 for a single sensor."""
    url = f'{OPENAQ_BASE}/sensors/{sensor_id}/measurements'
    params = {
        'period_name': 'day',
        'date_from':   f'{date_from}T00:00:00Z',
        'date_to':     f'{date_to}T23:59:59Z',
        'limit':       500,
    }
    resp = requests.get(url, params=params, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    return resp.json()

In [5]:
for key, cfg in CITIES_AQ.items():
    out_path = RAW_AQ / f'pm25_{key}_2025Q1.json'

    if out_path.exists():
        print(f'[{cfg["label"]}] file already exists – loading from disk.')
        with open(out_path) as f:
            data = json.load(f)
    else:
        print(f'[{cfg["label"]}] fetching from OpenAQ API …')
        data = fetch_openaq_daily(cfg['sensor_id'], DATE_FROM, DATE_TO)
        with open(out_path, 'w') as f:
            json.dump(data, f, indent=4)
        time.sleep(1)   # be polite to the API

    n = len(data.get('results', []))
    print(f'  → {n} daily records saved to {out_path.name}\n')

[New York City] file already exists – loading from disk.
  → 90 daily records saved to pm25_nyc_2025Q1.json

[Los Angeles] file already exists – loading from disk.
  → 90 daily records saved to pm25_la_2025Q1.json

[Chicago] file already exists – loading from disk.
  → 90 daily records saved to pm25_chicago_2025Q1.json



## 2. Open-Meteo Historical Weather API – Hourly Data

**API endpoint:** `https://archive-api.open-meteo.com/v1/archive`

Variables retrieved (hourly):
- `temperature_2m` – air temperature at 2 m (°C)
- `windspeed_10m` – wind speed at 10 m (km/h)
- `precipitation` – hourly precipitation sum (mm)

Coordinates for each city (WGS84):

In [6]:
OPENMETEO_URL = 'https://archive-api.open-meteo.com/v1/archive'

CITIES_WX = {
    'nyc':     {'lat': 40.7128, 'lon': -74.0060,  'label': 'New York City', 'tz': 'America/New_York'},
    'la':      {'lat': 34.0522, 'lon': -118.2437, 'label': 'Los Angeles',   'tz': 'America/Los_Angeles'},
    'chicago': {'lat': 41.8781, 'lon': -87.6298,  'label': 'Chicago',       'tz': 'America/Chicago'},
}

In [7]:
def fetch_openmeteo_hourly(lat: float, lon: float, tz: str,
                            date_from: str, date_to: str) -> dict:
    """Fetch hourly weather from the Open-Meteo Historical API."""
    params = {
        'latitude':   lat,
        'longitude':  lon,
        'start_date': date_from,
        'end_date':   date_to,
        'hourly':     'temperature_2m,windspeed_10m,precipitation',
        'timezone':   tz,
    }
    resp = requests.get(OPENMETEO_URL, params=params, timeout=30)
    resp.raise_for_status()
    return resp.json()

In [8]:
for key, cfg in CITIES_WX.items():
    out_path = RAW_WX / f'weather_{key}_2025Q1.json'

    if out_path.exists():
        print(f'[{cfg["label"]}] file already exists – loading from disk.')
        with open(out_path) as f:
            data = json.load(f)
    else:
        print(f'[{cfg["label"]}] fetching from Open-Meteo API …')
        data = fetch_openmeteo_hourly(
            cfg['lat'], cfg['lon'], cfg['tz'], DATE_FROM, DATE_TO
        )
        data['_city'] = key
        with open(out_path, 'w') as f:
            json.dump(data, f, indent=4)
        time.sleep(0.5)

    n = len(data.get('hourly', {}).get('time', []))
    print(f'  → {n} hourly records saved to {out_path.name}\n')

[New York City] file already exists – loading from disk.
  → 2160 hourly records saved to weather_nyc_2025Q1.json

[Los Angeles] file already exists – loading from disk.
  → 2160 hourly records saved to weather_la_2025Q1.json

[Chicago] file already exists – loading from disk.
  → 2160 hourly records saved to weather_chicago_2025Q1.json



## 3. Row-count Summary

In [9]:
print('=== Air Quality (OpenAQ) – daily records per city ===')
for key, cfg in CITIES_AQ.items():
    path = RAW_AQ / f'pm25_{key}_2025Q1.json'
    with open(path) as f:
        d = json.load(f)
    n = len(d.get('results', []))
    print(f'  {cfg["label"]:20s} : {n:4d} rows')

print()
print('=== Weather (Open-Meteo) – hourly records per city ===')
for key, cfg in CITIES_WX.items():
    path = RAW_WX / f'weather_{key}_2025Q1.json'
    with open(path) as f:
        d = json.load(f)
    n = len(d.get('hourly', {}).get('time', []))
    print(f'  {cfg["label"]:20s} : {n:4d} rows')

=== Air Quality (OpenAQ) – daily records per city ===
  New York City        :   90 rows
  Los Angeles          :   90 rows
  Chicago              :   90 rows

=== Weather (Open-Meteo) – hourly records per city ===
  New York City        : 2160 rows
  Los Angeles          : 2160 rows
  Chicago              : 2160 rows
